In [12]:
from openai import OpenAI
import json
from dotenv import load_dotenv
import os
from notion_client import Client
from utils import write_to_notion, format_result_to_markdown

load_dotenv()

# OpenAI setup
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define the structured output schema as JSON Schema
response_schema = {
    "format": {
    "type": "json_schema",
    "name": "document_summary",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "The document title"
            },
            "abstract": {
                "type": "string",
                "description": "The document abstract"
            },
            "summary": {
                "type": "string",
                "description": "Most critical information in brief bullet points"
            },
            "extended_summary": {
                "type": "string",
                "description": "Detailed explanation with relevant context"
            }
        },
        "required": ["title", "abstract", "summary", "extended_summary"],
        "additionalProperties": False
        }
    }
}


# gpt_model = "gpt-4o-mini"
gpt_model = "gpt-5-mini"

# Define the prompt template
prompt = """You are a concise document summariser. Read the PDF provided and
            return a clear, structured summary with key points, important details, and conclusions.
            Then provide an extended summary with relevant context which expands more on the main points."""

# Notion setup
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
DATABASE_ID = os.getenv("NOTION_DATABASE_ID")
notion = Client(auth=os.environ["NOTION_TOKEN"])

In [13]:
# put PDF URLs here
urls = [
    # "https://arxiv.org/pdf/2601.16120"
    "https://arxiv.org/pdf/2501.00663"
]

In [ ]:
for i, url in enumerate(urls):
    try:
        print(f"Processing document {i+1}/{len(urls)}: {url}")

        # Summarise the document with structured output using client.responses
        response = client.responses.create(
            model=gpt_model,
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": prompt
                        },
                        {
                            "type": "input_file",
                            "file_url": url
                        }
                    ]
                }
            ],
            text=response_schema
        )

        # Parse the structured response
        result_data = json.loads(response.output[1].content[0].text)
        title = result_data['title']
        print(f"Extracted Title: {title}")

        # Write to Notion
        md_content = format_result_to_markdown(result_data)
        result = write_to_notion(
            title=title,
            url=url,
            content=md_content,
            notion_token=NOTION_TOKEN,
            database_id=DATABASE_ID,
            model_name=response.model
        )
        print(f"Page created successfully: {result['id']}")

    except Exception as e:
        print(f"❌ Error processing {url}: {type(e).__name__}: {e}")
        continue

print(f"\n✅ Processing complete!")

Processing document 1/1: https://arxiv.org/pdf/2501.00663
❌ Error processing https://arxiv.org/pdf/2501.00663: TypeError: 'NoneType' object is not subscriptable

✅ Processing complete!


In [24]:
result_data = json.loads(response.output[1].content[0].text)
title = result_data['title']
print(f"Extracted Title: {title}")

Extracted Title: Titans: Learning to Memorize at Test Time


In [28]:
md_content = format_result_to_markdown(result_data)

In [32]:
result = write_to_notion(
            title=title,
            url=url,
            content=md_content,
            notion_token=NOTION_TOKEN,
            database_id=DATABASE_ID,
            model_name=response.model
        )
print(f"Page created successfully: {result['id']}")

Page created successfully: 2fabe5eb-a41a-81ec-b2c2-f2bcc0d842ba
